### Install Required Packages

In [ ]:
%%capture
%pip install quantumrings-toolkit-qiskit
%pip install qiskit

In [1]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#

import os
my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "scarlet_quantum_rings"

In [2]:
# Import from Quantum Rings Toolkit
from quantumrings.toolkit.qiskit import QrRuntimeService

# Acquire Quantum Rings backend
qr_services = QrRuntimeService(name = my_name, token = my_token)
qr_backend = qr_services.backend(name = my_backend, precision = "single")

In [3]:
from qiskit.circuit import (
    Parameter, QuantumCircuit, ClassicalRegister, QuantumRegister
)

#from qiskit.primitives import StatevectorSampler
from quantumrings.toolkit.qiskit import QrStatevectorSampler

import matplotlib.pyplot as plt
import numpy as np
 
# Define our circuit registers, including classical registers
# called 'alpha' and 'beta'.
qreg = QuantumRegister(3)
alpha = ClassicalRegister(2, "alpha")
beta = ClassicalRegister(1, "beta")
 
# Define a quantum circuit with two parameters.
circuit = QuantumCircuit(qreg, alpha, beta)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(1, 2)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure([0, 1], alpha)
circuit.measure([2], beta)

number_of_params = 100

# Define a sweep over parameter values, where the second axis is over.
# the two parameters in the circuit.
params = np.vstack([
    np.linspace(-np.pi, np.pi, number_of_params),
    np.linspace(-4 * np.pi, 4 * np.pi, number_of_params)
]).T
 
# Instantiate a new statevector simulation based sampler object.
sampler = QrStatevectorSampler(backend = qr_backend)
 
# Start a job that will return shots for all 100 parameter value sets.
pub = (circuit, params)
job = sampler.run([pub], shots=256)
 
# Extract the result for the 0th pub (this example only has one pub).
result = job.result()[0]
 
# There is one BitArray object for each ClassicalRegister in the
# circuit. Here, we can see that the BitArray for alpha contains data
# for all 100 sweep points, and that it is indeed storing data for 2
# bits over 256 shots.
assert result.data.alpha.shape == (number_of_params,)
assert result.data.alpha.num_bits == 2
assert result.data.alpha.num_shots == 256
 
# We can work directly with a binary array in performant applications.
raw = result.data.alpha.array
 
# For small registers where it is anticipated to have many counts
# associated with the same bitstrings, we can turn the data from,
# for example, the 22nd sweep index into a dictionary of counts.
counts = result.data.alpha.get_counts(22)
 
# Or, convert into a list of bitstrings that preserve shot order.
bitstrings = result.data.alpha.get_bitstrings(22)
print(bitstrings)

['00', '00', '10', '00', '00', '10', '10', '00', '10', '10', '00', '10', '10', '00', '00', '10', '10', '10', '10', '00', '10', '00', '00', '00', '00', '10', '10', '00', '10', '10', '00', '00', '10', '10', '10', '10', '00', '10', '10', '00', '00', '10', '10', '00', '10', '10', '00', '00', '10', '10', '10', '10', '00', '10', '10', '00', '10', '10', '00', '00', '00', '00', '10', '00', '00', '10', '00', '10', '00', '00', '10', '10', '00', '00', '00', '10', '00', '10', '10', '00', '00', '10', '10', '10', '00', '10', '10', '10', '00', '00', '10', '10', '10', '00', '00', '10', '10', '10', '10', '10', '10', '10', '10', '00', '00', '00', '10', '10', '10', '10', '10', '00', '10', '10', '00', '10', '10', '10', '00', '10', '00', '00', '10', '10', '00', '10', '00', '10', '10', '10', '00', '00', '10', '00', '00', '00', '00', '10', '10', '00', '00', '00', '10', '10', '10', '00', '10', '10', '10', '00', '00', '00', '10', '10', '10', '00', '00', '10', '10', '00', '10', '10', '00', '10', '10', '10', '00